# Capítulo 8 — Erros sob controle: o rover não pode travar

## Missão
Em Marte não existe botão de reiniciar. Você vai tratar erros com `try / except / else / finally`, ser específico nas exceções, criar exceções próprias da missão e escrever um interpretador de comandos que **nunca** derruba o rover.

**No fim deste caderno você terá:** o módulo `gabarito_rover/comandos.py`, que interpreta comandos de texto da Terra com exceções próprias e registro de falhas.

**Leitura essencial (opcional, livro em inglês):**
- Handling Errors Gracefully · PDF p. 272
- Being Specific about Exceptions · PDF p. 273
- Using try . . . except . . . else . . . finally · PDF p. 279
- Raising Your Own Exceptions · PDF p. 281

**Gabarito:** a mesma trilha com o código pronto. Tente primeiro em `praticar/`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados da missão do rover)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote gabarito_rover/
os.makedirs("../gabarito_rover", exist_ok=True)
open("../gabarito_rover/__init__.py", "a").close()        # gabarito_rover/ é o rover que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: gabarito_rover")

## Ideia em 30 segundos
- `try` tenta o código; `except TipoDeErro` trata o erro.
- `else` roda só se não houve erro; `finally` roda **sempre** (é a hora de fechar ou limpar).
- Capture o erro mais específico possível. Um `except:` sem tipo esconde bugs.
- `raise` levanta um erro, inclusive os seus, que são classes que herdam de `Exception`.

### 8.1 **Tratando erros.** Converta para número os textos `["87", "abc", "", "42.5"]`. Trate o `ValueError` mostrando qual texto falhou, sem parar o laço.

In [ ]:
for texto in ["87", "abc", "", "42.5"]:
    try:
        print(float(texto))
    except ValueError:
        print(f"não é número: {texto!r}")

### 8.2 **Vários except.** Escreva `media_por_hora(leituras, hora)`, que pega a temperatura de um dicionário hora → temp e divide pela quantidade de leituras da hora (que pode ser 0). Trate `KeyError` e `ZeroDivisionError` com mensagens diferentes.

In [ ]:
temps = {"06:00": -71.5, "12:00": -12.9}
contagem = {"06:00": 1, "12:00": 0}

def media_por_hora(hora):
    try:
        return temps[hora] / contagem[hora]
    except KeyError:
        return f"sem leitura às {hora}"
    except ZeroDivisionError:
        return f"nenhuma leitura contada às {hora}"

for h in ("06:00", "12:00", "09:00"):
    print(h, media_por_hora(h))

### 8.3 **else e finally.** Simule abrir o canal de rádio: num `try`, leia uma posição de `LEITURAS` (índice 3 e depois 99). No `else`, mostre a leitura; no `finally`, mostre "canal fechado" em todos os casos.

In [ ]:
for i in (3, 99):
    try:
        leitura = LEITURAS[i]
    except IndexError:
        print(f"não existe a leitura {i}")
    else:
        print("leitura:", leitura)
    finally:
        print("canal fechado")

### 8.4 **Exceções próprias.** Crie `ErroDoRover(Exception)` e duas filhas: `BateriaInsuficiente` e `ComandoInvalido`. Escreva `andar(bateria, distancia)`, que levanta `BateriaInsuficiente` se o consumo (`0,01 × distância`) deixar menos de 30%. Capture pela classe **mãe**.

In [ ]:
class ErroDoRover(Exception):
    pass

class BateriaInsuficiente(ErroDoRover):
    pass

class ComandoInvalido(ErroDoRover):
    pass

def andar(bateria, distancia):
    restante = bateria - 0.01 * distancia
    if restante < 30:
        raise BateriaInsuficiente(f"restariam {restante:.1f}%")
    return restante

for dist in (1000, 6000):
    try:
        print("bateria depois:", andar(73, dist))
    except ErroDoRover as e:
        print(type(e).__name__, "->", e)

### 8.5 **Interpretador de comandos.** Escreva `comandos.py` com as exceções do passo anterior e a classe `Interpretador(bateria=100)`:
- `executar(texto)` aceita `ANDAR <cm>`, `GIRAR <graus>` e `FOTO`;
- comando desconhecido ou número inválido levantam `ComandoInvalido`; falta de bateria levanta `BateriaInsuficiente`;
- `executar_lote(linhas)` roda todos, **sem parar** em erro, e guarda em `self.falhas` as tuplas `(linha, tipo do erro, mensagem)`. Devolve quantos deram certo.

_A célula grava `gabarito_rover/comandos.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../gabarito_rover/comandos.py
"""Interpretador de comandos enviados da Terra, à prova de falhas."""


class ErroDoRover(Exception):
    pass


class BateriaInsuficiente(ErroDoRover):
    pass


class ComandoInvalido(ErroDoRover):
    pass


class Interpretador:
    def __init__(self, bateria: float = 100):
        self.bateria = bateria
        self.falhas: list[tuple] = []
        self.fotos = 0

    def executar(self, texto: str) -> str:
        partes = texto.strip().upper().split()
        if not partes:
            raise ComandoInvalido("comando vazio")
        nome, args = partes[0], partes[1:]
        if nome == "FOTO":
            self.fotos += 1
            return f"foto {self.fotos}"
        if nome in ("ANDAR", "GIRAR"):
            try:
                valor = float(args[0])
            except (IndexError, ValueError):
                raise ComandoInvalido(f"{nome} precisa de um número") from None
            if nome == "GIRAR":
                return f"girou {valor % 360:.0f} graus"
            if self.bateria - 0.01 * valor < 30:
                raise BateriaInsuficiente(f"bateria {self.bateria:.1f}% não basta para {valor:.0f} cm")
            self.bateria -= 0.01 * valor
            return f"andou {valor:.0f} cm, bateria {self.bateria:.1f}%"
        raise ComandoInvalido(f"comando desconhecido: {nome}")

    def executar_lote(self, linhas: list[str]) -> int:
        ok = 0
        for linha in linhas:
            try:
                print(self.executar(linha))
                ok += 1
            except ErroDoRover as e:
                self.falhas.append((linha, type(e).__name__, str(e)))
        return ok


### 8.6 **Teste o lote.** Rode a sequência `["ANDAR 2000", "GIRAR 450", "FOTO", "PULAR 3", "ANDAR muito", "ANDAR 6000", "", "FOTO"]` com bateria 80. Mostre quantos deram certo e as falhas numa tabela.

In [ ]:
from gabarito_rover.comandos import Interpretador
rover = Interpretador(bateria=80)
ok = rover.executar_lote(["ANDAR 2000", "GIRAR 450", "FOTO", "PULAR 3", "ANDAR muito", "ANDAR 6000", "", "FOTO"])
print(f"{ok} comandos executados")
for falha in rover.falhas:
    print(falha)

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
